# Project Mythos Kaggle Pipeline

This notebook loads the ARC Prize 2026 ARC-AGI-2 data from Kaggle, loads a Mythos solver, runs the prediction pipeline, and writes `/kaggle/working/submission.json`.

Default mode uses the `pipeline` solver. It follows the master-plan stage order: ingest → JEPA encode adapter → HRM-Text planning adapter → world-model simulation adapter → TTT/LoRA adaptation adapter → HRM L-module execution adapter → decode/output. Real-model stages are explicit placeholders today; execution falls back to the baseline executor until HRM prediction is wired.

## 1. Configuration

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

DATA_DIR = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2')
SPLIT = 'test'  # 'training', 'evaluation', or 'test'
SOLVER_NAME = 'pipeline'  # 'pipeline', 'baseline', 'fixture', or 'hrm'
MODEL_MODE = 'fallback'  # 'fallback' loads configured models; 'strict' requires every planned model
OUTPUT_PATH = Path('/kaggle/working/submission.json')
RUN_HRM_SMOKE = False

# Set these to load real model checkpoints into the plan-aligned pipeline.
# os.environ['IJEPA_REPO_DIR'] = '/kaggle/working/ijepa'
# os.environ['IJEPA_CHECKPOINT_PATH'] = '/kaggle/input/your-jepa-checkpoint/checkpoint.pt'
# os.environ['HRM_TEXT_REPO_DIR'] = '/kaggle/working/hrm-text'
# os.environ['HRM_TEXT_CHECKPOINT_PATH'] = '/kaggle/input/your-hrm-text-checkpoint/checkpoint.pt'
# os.environ['WORLD_MODEL_CHECKPOINT_PATH'] = '/kaggle/input/your-world-model/world_model.pt'
# os.environ['TTT_LORA_CHECKPOINT_PATH'] = '/kaggle/input/your-lora-adapters/lora.pt'
# os.environ['HRM_REPO_DIR'] = '/kaggle/working/HRM'
# os.environ['HRM_CHECKPOINT_PATH'] = '/kaggle/input/your-hrm-checkpoint/checkpoint.pt'

print('DATA_DIR =', DATA_DIR)
print('SPLIT =', SPLIT)
print('SOLVER_NAME =', SOLVER_NAME)
print('MODEL_MODE =', MODEL_MODE)
print('OUTPUT_PATH =', OUTPUT_PATH)

## 2. Import Project Mythos

In [ ]:
PROJECT_ROOT = Path.cwd()
candidate_src_dirs = [
    PROJECT_ROOT / 'src',
    Path('/kaggle/working/src'),
    Path('/kaggle/working/ARC-AGI-2/src'),
]
candidate_src_dirs.extend(Path('/kaggle/input').glob('*/src'))
candidate_src_dirs.extend(Path('/kaggle/input').glob('*/ARC-AGI-2/src'))

for src_dir in candidate_src_dirs:
    if src_dir.exists() and str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

from mythos.arc import load_challenges
from mythos.kaggle_run import resolve_challenge_path, resolve_solution_path
from mythos.metrics import score_files
from mythos.pipeline import PLAN_STAGE_ORDER
from mythos.solvers.factory import make_solver
from mythos.submission import load_submission, write_submission

print('Imported Mythos from:', Path(sys.modules['mythos'].__file__).resolve())

## 3. Load Kaggle ARC Data

In [ ]:
challenge_path = resolve_challenge_path(DATA_DIR, SPLIT)
solution_path = resolve_solution_path(DATA_DIR, SPLIT)

tasks = load_challenges(challenge_path)
train_examples = sum(len(task.train) for task in tasks.values())
test_items = sum(len(task.test) for task in tasks.values())

print('challenge_path =', challenge_path)
print('solution_path =', solution_path)
print('tasks =', len(tasks))
print('train_examples =', train_examples)
print('test_items =', test_items)
first_task_id = next(iter(tasks))
print('first_task_id =', first_task_id)
print('first_task_train_pairs =', len(tasks[first_task_id].train))
print('first_task_test_items =', len(tasks[first_task_id].test))

## 4. Load Solver / Model

In [ ]:
solver = make_solver(SOLVER_NAME, model_mode=MODEL_MODE)
print('Loaded solver:', solver.__class__.__name__)

if SOLVER_NAME == 'pipeline':
    print('Pipeline mode: master-plan stage order is wired explicitly.')
    print('PLAN_STAGE_ORDER =', ' -> '.join(PLAN_STAGE_ORDER))
elif SOLVER_NAME == 'baseline':
    print('Baseline mode: simple rule solver plus guaranteed valid fallback predictions.')
elif SOLVER_NAME == 'fixture':
    print('Fixture mode: simple rule solver only; it may fail on real Kaggle tasks.')
elif SOLVER_NAME == 'hrm':
    print('HRM mode: external HRM environment required; direct prediction wiring is not implemented yet.')

if hasattr(solver, 'pipeline'):
    print('model_registry =')
    print(json.dumps(solver.pipeline.model_registry.summary(), indent=2))

## 5. Run Pipeline and Write Submission

In [ ]:
started = time.perf_counter()
predictions = []

for index, task in enumerate(tasks.values(), start=1):
    prediction = solver.solve(task)
    predictions.append(prediction)
    if index <= 3 or index == len(tasks):
        print(f'{index}/{len(tasks)} solved: {task.id}')

write_submission(predictions, OUTPUT_PATH)
elapsed = time.perf_counter() - started

print('Wrote submission:', OUTPUT_PATH)
print('tasks_predicted =', len(predictions))
print('elapsed_seconds =', round(elapsed, 3))

if hasattr(solver, 'last_trace') and solver.last_trace is not None:
    print('last_pipeline_trace =')
    print(json.dumps(solver.last_trace.to_dict(), indent=2))

## 6. Validate Output and Score When Possible

In [ ]:
submission = load_submission(OUTPUT_PATH)
submission_items = sum(len(outputs) for outputs in submission.values())

print('submission_tasks =', len(submission))
print('submission_test_items =', submission_items)
print('sample_task_id =', next(iter(submission)))
print('sample_prediction =')
print(json.dumps(submission[next(iter(submission))][0].__dict__, indent=2))

if solution_path is not None:
    score = score_files(str(OUTPUT_PATH), str(solution_path))
    print('score =')
    print(json.dumps(score.to_dict(), indent=2, sort_keys=True))
else:
    print('No solutions file for this split; skipping score.')

## 7. Optional HRM Smoke Test

Set `RUN_HRM_SMOKE = True`, `HRM_REPO_DIR`, and `HRM_CHECKPOINT_PATH` in the configuration cell to verify that the external HRM checkout imports and the checkpoint loads. This does not yet run HRM predictions through `submission.json`.

In [ ]:
if RUN_HRM_SMOKE:
    from mythos.hrm_dataset import prepare_hrm_raw_dataset
    from mythos.solvers.hrm import HRMEnvironment

    env = HRMEnvironment.from_env()
    env.validate(require_cuda=True)
    modules = env.import_modules()
    checkpoint = env.load_checkpoint()
    raw_dir = prepare_hrm_raw_dataset(tasks.values(), Path('/kaggle/working/hrm_smoke/raw/ARC-AGI-2/data'))

    print('HRM repo:', env.repo_dir)
    print('HRM checkpoint:', env.checkpoint_path)
    print('Imported modules:', sorted(modules))
    print('Checkpoint type:', type(checkpoint).__name__)
    print('Prepared HRM raw data:', raw_dir)
else:
    print('RUN_HRM_SMOKE is False; skipping HRM smoke test.')